# Do proper emulation by aligning enrollment and treatment initiation

We will now align eligibility criterion/enrollment into the WHI observational study with treatment initiation. We will do so by excluding current users (and maybe past users?) of combined HRT.

In [1]:
import pandas as pd 
import numpy as np 
import os 
import sys 
from tqdm import tqdm
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from scipy.stats import zscore

In [2]:

# read tables
dir_path = '/Users/zeshanmh/Documents/research/benchmarking-os/'
out   = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/outc_adj_bio.csv'))
ct_fu = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/adh_ht_pub.csv'))[['ID', 'ADHRATE', 'ENDDY', 'STARTDY', 'LOST', 'STOPHRT']] 
std_trt = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/dem_ctos_bio.csv'))[['ID', 'HRTARM', 'OSFLAG']]


In [3]:
# List of outcomes     
glbl_list = ['CHD', 'BREAST', 'STROKE', 'PE', 'ENDMTRL', 'COLORECTAL', 'BKHIP', 'DEATH']    
other_list = ['PTCA', 'DVT']

In [4]:
# Get end of follow-up for CT patients 
# BTW, do we have to consider START-DAY? what about LOST for censoring?

# keep only those with ADHRATE not missing, and group by ID to get max ENDDY
# keep columns 'ID', 'ENDDY', and 'LOST'
# rename ENDDY to END_DY
# ct_end = ct_fu[ct_fu['ADHRATE'].notna()].groupby('ID')['ENDDY'].max().reset_index() 
# ct_end = ct_end.rename(columns={'ENDDY': 'END_DY'})
# ct_end

# ct_end = ct_fu[ct_fu['ADHRATE'].notna()][['ADHRATE','ID', 'ENDDY', 'LOST']].rename(columns={'ENDDY': 'END_DY'})
# ct_end = ct_fu[['ADHRATE','ID', 'ENDDY', 'LOST']].rename(columns={'ENDDY': 'END_DY'})
# ct_end = ct_end.query('ADHRATE != 0.')[['ID','END_DY','LOST']]
ct_end = ct_fu[ct_fu['ADHRATE'].notna()].groupby('ID')['ENDDY'].max().reset_index() 
ct_end = ct_end.rename(columns={'ENDDY': 'END_DY'})
ct_end

,ID,END_DY
0,500001,2190
1,500022,3287
2,500024,1095
3,500025,2190
4,500027,2921
...,...,...
27168,699951,2556
27169,699963,2556
27170,699974,1460
27171,699987,4017


In [5]:
ct_df = std_trt.drop_duplicates('ID')
ct_df = ct_df[ct_df['HRTARM'].isin(['E+P intervention', 'E+P control'])]
ct_df = ct_df.merge(ct_end, on='ID', how='left')
ct_df = ct_df.merge(out, on='ID', how='left')

# code variables HRTARM and OS 
ct_df['OS'] = 0 
ct_df['HRTARM'] = ct_df['HRTARM'].map({'E+P intervention': 1, 'E+P control': 0})

# print out first 10 rows
print(ct_df.shape)
print(ct_df[ct_df['HRTARM'] == 1].shape)
print(ct_df[ct_df['HRTARM'] == 0].shape)
ct_df.head(n=10)


(16608, 358)
(8506, 358)
(8102, 358)


,ID,HRTARM,OSFLAG,END_DY,ANGINA,ANGINADY,ANGINASRC,AANEUR,AANEURDY,AANEURSRC,...,DEATHDY,DEATHSRC,DEATHCAUSESRC,HYST,HYSTDY,HYSTSRC,ENDWHIDY,ENDEXT1DY,ENDFOLLOWDY,OS
0,642629,1,No,1460.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,1,4499.0,1.0,3480.0,5481.0,9083.0,0
1,568085,1,No,1825.0,0,NaN,NaN,0,NaN,NaN,...,7610.0,2.0,1.0,0,NaN,NaN,2725.0,4726.0,7610.0,0
2,568186,0,No,2555.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,3138.0,5139.0,8013.0,0
3,623255,1,No,1825.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,2500.0,4501.0,8015.0,0
4,537848,1,No,2555.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,3263.0,5264.0,8992.0,0
5,668539,1,No,2556.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,3486.0,5487.0,5487.0,0
6,660370,1,No,1095.0,0,NaN,NaN,0,NaN,NaN,...,827.0,1.0,1.0,0,NaN,NaN,827.0,827.0,827.0,0
7,551999,1,No,1094.0,0,NaN,NaN,0,NaN,NaN,...,3926.0,1.0,1.0,0,NaN,NaN,3012.0,3926.0,3926.0,0
8,682433,0,No,1095.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,1,834.0,0.0,2521.0,4522.0,8021.0,0
9,605069,1,No,1460.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,2681.0,4682.0,8375.0,0


In [6]:
# process outcomes 
for i in glbl_list + other_list: 
    ct_df[i+'_E']  = ((ct_df[i] == 1) & (ct_df[i+'DY'] <= ct_df['END_DY'])).astype(int)
    ct_df[i+'_DY'] = np.where(ct_df[i+'_E'] == 1, ct_df[i+'DY'], ct_df['END_DY'])
    ct_df[i+'_EDY'] = np.where(ct_df[i+'_E'] == 1, ct_df[i+'DY'], np.nan) 

# Global index
ct_df['GLBL_E'] = (ct_df[[j+'_E' for j in glbl_list]].sum(axis=1) > 0).astype(int)
ct_df['GLBL_DY'] = np.where(ct_df['GLBL_E'] == 1,
                            ct_df[[j+'_EDY' for j in glbl_list]].min(axis=1),
                            ct_df[[j+'_DY' for j in glbl_list]].min(axis=1))

# Select needed columns
ct_df = ct_df[['ID', 'OS', 'HRTARM'] + 
                [j+'_E' for j in glbl_list + other_list + ['GLBL']] + 
                [j+'_DY' for j in glbl_list + other_list + ['GLBL']]]


In [7]:
ct_df.query('HRTARM == 0 & CHD_E == 1')

,ID,OS,HRTARM,CHD_E,BREAST_E,STROKE_E,PE_E,ENDMTRL_E,COLORECTAL_E,BKHIP_E,...,BREAST_DY,STROKE_DY,PE_DY,ENDMTRL_DY,COLORECTAL_DY,BKHIP_DY,DEATH_DY,PTCA_DY,DVT_DY,GLBL_DY
21,603833,0,0,1,0,0,1,0,0,0,...,729.0,729.0,538.0,729.0,729.0,729.0,729.0,729.0,729.0,508.0
31,592726,0,0,1,0,0,0,0,0,0,...,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2036.0
78,602131,0,0,1,0,0,0,0,0,0,...,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,2180.0
275,592897,0,0,1,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2097.0,2190.0,2190.0,2097.0
292,520105,0,0,1,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2102.0,2190.0,2190.0,2102.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16066,614132,0,0,1,0,0,0,0,0,0,...,1826.0,1826.0,1826.0,1826.0,1826.0,1826.0,1593.0,1826.0,1826.0,1593.0
16203,678820,0,0,1,0,0,0,0,0,0,...,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,912.0,2555.0,912.0
16234,664063,0,0,1,0,0,0,0,0,0,...,2556.0,2556.0,2556.0,2556.0,2556.0,2556.0,2556.0,1342.0,2556.0,1342.0
16249,597604,0,0,1,0,0,0,0,0,0,...,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,1913.0,2191.0,2191.0,1913.0


In [8]:
dir_path = '/Users/zeshanmh/Documents/research/benchmarking-os/'
hyst    = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f2_ctos_bio.csv'))[['ID','HYST']]
pre_hrt  = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f43_ctos_bio.csv'))[['ID', 'TOTESTAT','TOTPSTAT']]
post_hrt = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f48_av1_os_pub.csv'))[['ID','ELSTYR','PLSTYR','HRTCMBP']]

/var/folders/t9/9775q6dn21l67f71h7t7xj0h0000gn/T/ipykernel_68692/907246142.py:2: DtypeWarning: Columns (39,42,57,59,66) have mixed types. Specify dtype option on import or set low_memory=False.
  hyst    = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f2_ctos_bio.csv'))[['ID','HYST']]


In [9]:
# construct os_df 
os_df = std_trt.drop_duplicates('ID')
os_df = os_df[os_df['OSFLAG'] == 'Yes']
os_df = os_df.merge(hyst, on='ID', how='left')
os_df = os_df[os_df['HYST'] == 'No']
os_df = os_df.merge(pre_hrt, on='ID', how='left')
print(os_df['TOTESTAT'].value_counts())
os_df = os_df[os_df['TOTESTAT'].isin(['Never used', 'Past user'])]
os_df = os_df.merge(post_hrt, on='ID', how='left')
os_df = os_df.merge(out, on='ID', how='left')

# 35551 (control) + 17503 (intervention) = 53054
print(os_df[os_df['TOTPSTAT'].isin(['Current user', 'Never used', 'Past user'])].shape)
print('Dropping current users of combined HRT')
print(os_df[os_df['TOTPSTAT'].isin(['Never used', 'Past user'])].shape)
os_df = os_df[os_df['TOTPSTAT'].isin(['Never used', 'Past user'])]
conditions = [
    (((os_df['ELSTYR'] == 'Yes') & (os_df['PLSTYR'] == 'Yes')) | (os_df['HRTCMBP'] == 'Yes')),
    ((os_df['ELSTYR'] == 'No') & (os_df['PLSTYR'] == 'No')),
    (((os_df['ELSTYR'] == 'Yes') & (os_df['PLSTYR'] == 'No')) | ((os_df['ELSTYR'] == 'No') & (os_df['PLSTYR'] == 'Yes')))
]
choices = [1, 0, -1]
os_df['HRTGRP'] = np.select(conditions, choices, default=-2)
os_df = os_df[os_df['HRTGRP'] != -2]
os_df['HRTARM'] = (os_df['HRTGRP'] == 1).astype(int)
os_df['OS'] = 1

# os_end_day = None
os_end_day = 6*365
os_df['END_DY'] = os_end_day if os_end_day is not None else os_df['ENDFOLLOWDY']
# os_df['END_DY'] = os_df.apply(lambda x: x['DEATHDY'] if x['DEATHDY'] < os_end_day else os_end_day, axis=1)

# Process outcomes (same as CT)
for i in glbl_list + other_list:
    os_df[i+'_E'] = ((os_df[i] == 1) & (os_df[i+'DY'] <= os_df['END_DY'])).astype(int)
    os_df[i+'_DY'] = np.where(os_df[i+'_E'] == 1, os_df[i+'DY'], os_df['END_DY'])
    os_df[i+'_EDY'] = np.where(os_df[i+'_E'] == 1, os_df[i+'_DY'], np.nan)

# Global index
os_df['GLBL_E'] = (os_df[[j+'_E' for j in glbl_list]].sum(axis=1) > 0).astype(int)
os_df['GLBL_DY'] = np.where(os_df['GLBL_E'] == 1,
                            os_df[[j+'_EDY' for j in glbl_list]].min(axis=1),
                            os_df[[j+'_DY' for j in glbl_list]].min(axis=1))

# Select needed columns
os_df = os_df[['ID', 'OS', 'HRTARM'] + 
                [j+'_E' for j in glbl_list + other_list + ['GLBL']] + 
                [j+'_DY' for j in glbl_list + other_list + ['GLBL']]]

os_df


TOTESTAT
Never used      47832
Past user        5244
Current user     1343
Name: count, dtype: int64
(53048, 362)
Dropping current users of combined HRT
(35539, 362)


,ID,OS,HRTARM,CHD_E,BREAST_E,STROKE_E,PE_E,ENDMTRL_E,COLORECTAL_E,BKHIP_E,...,BREAST_DY,STROKE_DY,PE_DY,ENDMTRL_DY,COLORECTAL_DY,BKHIP_DY,DEATH_DY,PTCA_DY,DVT_DY,GLBL_DY
0,591800,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
1,548198,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
4,508135,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
6,626126,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
8,571734,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53064,622001,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
53065,684946,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
53066,584010,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
53071,641028,1,1,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0


In [10]:
# check outcome of breast cancer and CHD in both ct and os
print('Clinical Trial')
print(f'No. of women in control arm: {ct_df[ct_df["HRTARM"] == 0].shape[0]}')
print(f'No. of women in intervention arm: {ct_df[ct_df["HRTARM"] == 1].shape[0]}')
print(f'No. of CHD events in control arm: {ct_df.query("HRTARM == 0 & CHD_E == 1").shape[0]}')
print(f'No. of CHD events in intervention arm: {ct_df.query("HRTARM == 1 & CHD_E == 1").shape[0]}')
print(f'No. of breast cancer events in control arm: {ct_df.query("HRTARM == 0 & BREAST_E == 1").shape[0]}')
print(f'No. of breast cancer events in intervention arm: {ct_df.query("HRTARM == 1 & BREAST_E == 1").shape[0]}')
print(f'No. of stroke events in control arm: {ct_df.query("HRTARM == 0 & STROKE_E == 1").shape[0]}')
print(f'No. of stroke events in intervention arm: {ct_df.query("HRTARM == 1 & STROKE_E == 1").shape[0]}')
print()
print()

print('Observational Study')
print(f'No. of women in control arm: {os_df[os_df["HRTARM"] == 0].shape[0]}')
print(f'No. of women in intervention arm: {os_df[os_df["HRTARM"] == 1].shape[0]}')
print(f'No. of CHD events in control arm: {os_df.query("HRTARM == 0 & CHD_E == 1").shape[0]}')
print(f'No. of CHD events in intervention arm: {os_df.query("HRTARM == 1 & CHD_E == 1").shape[0]}')
print(f'No. of breast cancer events in control arm: {os_df.query("HRTARM == 0 & BREAST_E == 1").shape[0]}')
print(f'No. of breast cancer events in intervention arm: {os_df.query("HRTARM == 1 & BREAST_E == 1").shape[0]}')
print(f'No. of stroke events in control arm: {os_df.query("HRTARM == 0 & STROKE_E == 1").shape[0]}')
print(f'No. of stroke events in intervention arm: {os_df.query("HRTARM == 1 & STROKE_E == 1").shape[0]}')


Clinical Trial
No. of women in control arm: 8102
No. of women in intervention arm: 8506
No. of CHD events in control arm: 129
No. of CHD events in intervention arm: 169
No. of breast cancer events in control arm: 169
No. of breast cancer events in intervention arm: 231
No. of stroke events in control arm: 88
No. of stroke events in intervention arm: 124


Observational Study
No. of women in control arm: 28885
No. of women in intervention arm: 3404
No. of CHD events in control arm: 552
No. of CHD events in intervention arm: 51
No. of breast cancer events in control arm: 881
No. of breast cancer events in intervention arm: 122
No. of stroke events in control arm: 421
No. of stroke events in intervention arm: 46


In [11]:
# Combine CT and OS data
ctos_df = pd.concat([ct_df, os_df], ignore_index=True)


In [12]:
ctos_df

,ID,OS,HRTARM,CHD_E,BREAST_E,STROKE_E,PE_E,ENDMTRL_E,COLORECTAL_E,BKHIP_E,...,BREAST_DY,STROKE_DY,PE_DY,ENDMTRL_DY,COLORECTAL_DY,BKHIP_DY,DEATH_DY,PTCA_DY,DVT_DY,GLBL_DY
0,642629,0,1,0,0,0,1,0,0,0,...,1460.0,1460.0,935.0,1460.0,1460.0,1460.0,1460.0,1460.0,936.0,935.0
1,568085,0,1,0,0,0,0,0,0,0,...,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0
2,568186,0,0,0,0,0,0,0,0,0,...,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0
3,623255,0,1,0,0,0,0,0,0,0,...,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0
4,537848,0,1,0,0,0,0,0,0,0,...,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48892,622001,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
48893,684946,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
48894,584010,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
48895,641028,1,1,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0


# Analysis

In [13]:
import pandas.api.types as ptypes

ctos_temp = ctos_df.copy()
# Dictionary to specify which features are categorical
categorical_features = {
    'dem_ctos_bio.csv': {'AGE': False, 'ETHNIC': True, 'EDUC': True}, 
    'f80_ctos_bio.csv': {'BMI': False}, 
    'f34_ctos_bio.csv': {'SMOKING': True}, 
    'f31_ctos_bio.csv': {'MENO': False}, 
    'f151_ctos_bio.csv': {'PHYSFUN': False}    
}

new_feature_dict = { 
    'dem_ctos_bio.csv': ['AGE', 'ETHNIC_White', \
                         'EDUC_Some post-graduate or professional', \
                         'EDUC_Some college or Associate Degree'],
    'f80_ctos_bio.csv': ['BMI'],
    'f34_ctos_bio.csv': ['SMOKING_Past Smoker', 'SMOKING_Current Smoker'],
    'f31_ctos_bio.csv': ['MENO'],
    'f151_ctos_bio.csv': ['PHYSFUN']
}

# dfs = []  # Store all dataframes to concatenate later
new_dir_path = dir_path + 'whi/data/data/main_study/csv'

for filename, f_dict in categorical_features.items():
    # Read the data
    df = pd.read_csv(os.path.join(new_dir_path, filename))
    if filename == 'f80_ctos_bio.csv': 
        df = df.query('F80VTYP == "Screening"')
    elif filename == 'f151_ctos_bio.csv': 
        idx = df.groupby('ID')['F151DAYS'].idxmin().reset_index(drop=True)
        df = df.loc[idx, :].reset_index(drop=True)[['ID','PHYSFUN']]
    # Select needed columns
    features = list(f_dict.keys())
    df = df[['ID'] + features]
    
    # Separate ID column
    id_col = df['ID']
    print(f"Processed {filename}")
    print(df.shape)

    orig_cols = ctos_temp.columns.tolist()
    ctos_temp = ctos_temp.merge(df, on='ID', how='left')

    # Handle continuous and categorical features separately
    cont_features = [f for f in features if not f_dict[f]]
    cat_features = [f for f in features if f_dict[f]]
    
    # Handle continuous features
    if cont_features:
        cont_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
        ctos_temp[cont_features] = cont_imputer.fit_transform(ctos_temp[cont_features])
    
    # Handle categorical features
    if cat_features:
        cat_imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        ctos_temp[cat_features] = cat_imputer.fit_transform(ctos_temp[cat_features])
        
        # One-hot encode categorical features
        ctos_temp = pd.get_dummies(ctos_temp, columns=cat_features, prefix=cat_features)

    if filename == 'dem_ctos_bio.csv': 
        ctos_temp = ctos_temp.rename(columns={'ETHNIC_White (not of Hispanic origin)': 'ETHNIC_White'})

    ctos_temp = ctos_temp[orig_cols + new_feature_dict[filename]]

ctos_temp = ctos_temp.astype({col: int for col in ctos_temp.select_dtypes(include='bool').columns})
display(ctos_temp)    


Processed dem_ctos_bio.csv
(161808, 4)
Processed f80_ctos_bio.csv
(161771, 2)
Processed f34_ctos_bio.csv
(161625, 2)


/var/folders/t9/9775q6dn21l67f71h7t7xj0h0000gn/T/ipykernel_68692/2535602248.py:28: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(new_dir_path, filename))


Processed f31_ctos_bio.csv
(161705, 2)
Processed f151_ctos_bio.csv
(113491, 2)


,ID,OS,HRTARM,CHD_E,BREAST_E,STROKE_E,PE_E,ENDMTRL_E,COLORECTAL_E,BKHIP_E,...,GLBL_DY,AGE,ETHNIC_White,EDUC_Some post-graduate or professional,EDUC_Some college or Associate Degree,BMI,SMOKING_Past Smoker,SMOKING_Current Smoker,MENO,PHYSFUN
0,642629,0,1,0,0,0,1,0,0,0,...,935.0,64.0,1,0,0,29.19411,0,0,54.0,65.000000
1,568085,0,1,0,0,0,0,0,0,0,...,1825.0,62.0,0,0,0,19.55943,0,1,51.0,90.000000
2,568186,0,0,0,0,0,0,0,0,0,...,2555.0,62.0,1,0,1,30.44928,1,0,44.0,50.000000
3,623255,0,1,0,0,0,0,0,0,0,...,1825.0,60.0,1,1,0,28.54828,0,0,54.0,76.676011
4,537848,0,1,0,0,0,0,0,0,0,...,2555.0,54.0,1,0,1,40.32766,1,0,54.0,65.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48892,622001,1,0,0,0,0,0,0,0,0,...,2190.0,58.0,1,0,0,23.02576,1,0,50.0,90.000000
48893,684946,1,0,0,0,0,0,0,0,0,...,2190.0,55.0,1,0,0,29.10326,0,0,45.0,95.000000
48894,584010,1,0,0,0,0,0,0,0,0,...,2190.0,62.0,1,0,0,20.94698,1,0,49.0,100.000000
48895,641028,1,1,0,0,0,0,0,0,0,...,2190.0,69.0,1,1,0,25.50301,0,0,54.0,90.000000


In [14]:
# hazard ratios for stroke, breast cancer, and CHD in clinical trial vs observational study 

## CT 
ct_df = ctos_temp.query('OS == 0')
ct_df_sub = ct_df[['ID','HRTARM', 'STROKE_E', 'BREAST_E', 'CHD_E','STROKE_DY', 'BREAST_DY', 'CHD_DY']]
ct_df_chd = ct_df[['HRTARM', 'CHD_E', 'CHD_DY']]
ct_df_chd = ct_df_chd[ct_df_chd['CHD_DY'].notna()]

ct_df_stroke = ct_df[['HRTARM', 'STROKE_E', 'STROKE_DY']]
ct_df_stroke = ct_df_stroke[ct_df_stroke['STROKE_DY'].notna()]

ct_df_breast = ct_df[['HRTARM', 'BREAST_E', 'BREAST_DY']]
ct_df_breast = ct_df_breast[ct_df_breast['BREAST_DY'].notna()]

from lifelines import CoxPHFitter

def get_hr(df, Y, E, event_name, HR_cov='HRTARM', study_type='Clinical Trial'): 
    cph = CoxPHFitter()
    cph.fit(df, duration_col=Y, event_col=E)
    cph.print_summary()
    cHR = cph.hazard_ratios_[HR_cov]
    cis = cph.confidence_intervals_
    lower = np.exp(cis['95% lower-bound'][HR_cov])
    upper = np.exp(cis['95% upper-bound'][HR_cov])
    print(f'Hazard ratio for {event_name} in {study_type}: {np.round(cHR, 2)} (95% CI: {np.round(lower, 2)}, {np.round(upper, 2)})')

get_hr(ct_df_chd, 'CHD_DY', 'CHD_E', 'CHD')
get_hr(ct_df_stroke, 'STROKE_DY', 'STROKE_E', 'Stroke')
get_hr(ct_df_breast, 'BREAST_DY', 'BREAST_E', 'Breast Cancer')


<lifelines.CoxPHFitter: fitted with 16516 total observations, 16218 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 16516
number of events observed = 298
   partial log-likelihood = -2787.04
         time fit was run = 2025-01-09 18:26:26 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
HRTARM     0.24      1.28      0.12            0.01            0.47                1.01                1.61

           cmp to    z    p  -log2(p)
covariate                            
HRTARM       0.00 2.09 0.04      4.76
---
Concordance = 0.54
Partial AIC = 5576.08
log-likelihood ratio test = 4.39 on 1 df
-log2(p) of ll-ratio test = 4.79

Hazard ratio for CHD in Clinical Trial: 1.28 (95% CI: 1.01, 1.61)


<lifelines.CoxPHFitter: fitted with 16516 total observations, 16304 right-censored observations>
             duration col = 'STROKE_DY'
                event col = 'STROKE_E'
      baseline estimation = breslow
   number of observations = 16516
number of events observed = 212
   partial log-likelihood = -1970.95
         time fit was run = 2025-01-09 18:26:27 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
HRTARM     0.32      1.37      0.14            0.04            0.59                1.04                1.80

           cmp to    z    p  -log2(p)
covariate                            
HRTARM       0.00 2.27 0.02      5.41
---
Concordance = 0.53
Partial AIC = 3943.90
log-likelihood ratio test = 5.21 on 1 df
-log2(p) of ll-ratio test = 5.47

Hazard ratio for Stroke in Clinical Trial: 1.37 (95% CI: 1.04, 1.8)


<lifelines.CoxPHFitter: fitted with 16516 total observations, 16116 right-censored observations>
             duration col = 'BREAST_DY'
                event col = 'BREAST_E'
      baseline estimation = breslow
   number of observations = 16516
number of events observed = 400
   partial log-likelihood = -3713.55
         time fit was run = 2025-01-09 18:26:27 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
HRTARM     0.29      1.34      0.10            0.09            0.49                1.09                1.63

           cmp to    z      p  -log2(p)
covariate                              
HRTARM       0.00 2.86 <0.005      7.86
---
Concordance = 0.52
Partial AIC = 7429.11
log-likelihood ratio test = 8.25 on 1 df
-log2(p) of ll-ratio test = 7.94

Hazard ratio for Breast Cancer in Clinical Trial: 1.34 (95% CI: 1.09, 1.63)


In [15]:
# OS 
os_df = ctos_temp.query('OS == 1')
features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'

os_df_sub = os_df[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]

get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')




<lifelines.CoxPHFitter: fitted with 32289 total observations, 31686 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 32289
number of events observed = 603
   partial log-likelihood = -6100.98
         time fit was run = 2025-01-09 18:26:38 UTC

---
                                         coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                
AGE                                      0.09      1.10      0.01            0.08            0.11                1.08                1.11
ETHNIC_White                             0.12      1.13      0.13           -0.13            0.37                0.88                1.45
EDUC_Some post-graduate or professional -0.21      0.81      0.14           -0.47            0.06                0.62                1.06
EDUC_Some college or Associate Degree   -0.26      0.77      0.10           -0.45           -0.07                0.63                0.94
BMI                                      0.04      1.04      0.01            0.03            0.05                1.03                1.05
SMOKING_Past Smoker                      0.25      1.28      0.09            0.08            0.41                1.08                1.51
SMOKING_Current Smoker                   0.50      1.65      0.16            0.18            0.82                1.20                2.27
MENO                                    -0.02      0.98      0.01           -0.04           -0.01                0.96                0.99
PHYSFUN                                 -0.00      1.00      0.00           -0.01           -0.00                0.99                1.00
HRTARM                                   0.09      1.09      0.15           -0.20            0.38                0.82                1.46

                                         cmp to     z      p  -log2(p)
covariate                                                             
AGE                                        0.00 14.04 <0.005    146.39
ETHNIC_White                               0.00  0.94   0.35      1.52
EDUC_Some post-graduate or professional    0.00 -1.50   0.13      2.91
EDUC_Some college or Associate Degree      0.00 -2.63   0.01      6.85
BMI                                        0.00  6.56 <0.005     34.16
SMOKING_Past Smoker                        0.00  2.88 <0.005      8.00
SMOKING_Current Smoker                     0.00  3.06 <0.005      8.82
MENO                                       0.00 -2.83 <0.005      7.76
PHYSFUN                                    0.00 -2.33   0.02      5.65
HRTARM                                     0.00  0.59   0.56      0.84
---
Concordance = 0.70
Partial AIC = 12221.95
log-likelihood ratio test = 308.01 on 10 df
-log2(p) of ll-ratio test = 197.66

Hazard ratio for CHD in Observational Study: 1.09 (95% CI: 0.82, 1.46)
